# KOS Pipeline Demo
### Automated Knowledge Organisation for the National Library Board

**The problem:** KOS librarians at NLB maintain authoritative entity records in TTE (Thesaurus and Taxonomy Editor) — people, organisations, places, buildings, awards, events. They currently monitor news manually, copy-paste names into TTE to check for existing records, and update fields by hand. 2–3 hours per week per person.

**The solution:** A semi-automated pipeline that reads Singapore news articles and proposes structured, evidence-backed updates to TTE for human review and approval.

**One line:** In the past you had to read news because it was part of your job. Now you read it because you want to.

---

| Step | Script | What | LLM? |
|------|--------|------|------|
| 0 | *(done)* | Scrape articles + classify event types | Yes |
| 1 | `01_build_index.py` | Build BM25 alias index from TTE CSVs | No |
| 2 | `build_prompts.py` | Build one fact-extraction prompt per article × event type | No |
| 3 | *(done)* | Send prompts to LLM, save responses | Yes |
| 4 | `join_responses.py` | BM25 link + TTE lookup + merge state → review queue | No |
| 5 | `review_app.py` | Human review — approve / reject / flag / export | No |

**No TTE writes without human approval. Every proposed change traces to an exact article sentence.**

---
*Steps 0, 1, and 3 are pre-done. This notebook runs Steps 2 and 4 live, then walks one article all the way from raw text to review card.*

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import os
import pickle
import subprocess

import pandas as pd

import config

print('Setup complete.')

## Step 0 — Input data

2,970 CNA articles scraped and pre-processed with two LLM calls (already done):

**Classifier (Prompt 1):** Each article *title* is sent to the LLM with a classifier prompt. Returns an event type from a 37-type taxonomy, or "not relevant". Cheap gate — filters ~85% of articles before spending on full text.

Results are in `cna_articles.csv`. Columns used downstream: `Title`, `URL`, `text`, `Event_Type`, `Singapore_Context`.

In [ ]:
articles = pd.read_csv('../data/cna_articles.csv', dtype=str).fillna('')
relevant = articles[
    articles['Event_Type'].str.strip().str.lower().ne('not relevant') &
    articles['Singapore_Context'].str.strip().str.lower().eq('true')
]

print(f'Total articles scraped : {len(articles):,}')
print(f'Relevant articles      : {len(relevant):,}')
print(f'Relevance rate         : {len(relevant)/len(articles)*100:.1f}%')
print()
print('Top event types:')
print(relevant['Event_Type'].value_counts().head(10).to_string())

## Step 1 — BM25 entity search index

TTE contains ~55,000 entities across 8 vocabularies. Each entity has multiple alias surfaces — authorised name, display name, UF variants ("Used For": acronyms, former names, alternate spellings), and cross-language forms (English ↔ Chinese ↔ Malay ↔ Tamil).

We build a BM25 full-text search index over all these surfaces. When `join_responses.py` sees "URA", the index returns "Urban Redevelopment Authority (Singapore)".

**Why BM25 not vector search:** Entity queries from news are proper noun surface strings. "Lee Kuan Yew" matching "Lee Kuan Yew" is a lexical problem, not a semantic one. BM25 is faster, simpler, and more accurate for this retrieval pattern.

In [ ]:
with open('../outputs/bm25_index.pkl', 'rb') as f:
    index = pickle.load(f)

print('BM25 index loaded.\n')
for vocab, vi in index.items():
    n_entities = len(set(m['uid'] for m in vi['metadata']))
    n_surfaces = len(vi['surfaces'])
    print(f'  {vocab:<15} : {n_entities:>6,} entities,  {n_surfaces:>6,} alias surfaces')

In [ ]:
# Quick BM25 search helper (mirrors logic in join_responses.py)
def bm25_search(index, entity_name, vocab, top_k=3):
    import re
    if vocab not in index:
        return []
    vi = index[vocab]
    tokens = [t for t in re.split(r'[^\w]+', entity_name.lower()) if len(t) >= 2]
    if not tokens:
        return []
    scores = vi['bm25'].get_scores(tokens)
    top_idx = scores.argsort()[::-1][:top_k]
    return [
        {'authorised_name': vi['metadata'][i]['authorised_name'],
         'uid':             vi['metadata'][i]['uid'],
         'score':           round(float(scores[i]), 4)}
        for i in top_idx if scores[i] > 0
    ]

# Exact name — high match
print("Query: 'Adele Tan Shiao Ling' in PERSON")
for r in bm25_search(index, 'Adele Tan Shiao Ling', 'PERSON'):
    tag = '  ← HIGH MATCH' if r['score'] >= config.HIGH_MATCH_THRESHOLD else ''
    print(f"  {r['score']:>8.4f}  {r['authorised_name']}  (UID: {r['uid']}){tag}")

print()

# Acronym resolved via UF index
print("Query: 'URA' in ORGANISATION")
for r in bm25_search(index, 'URA', 'ORGANISATION'):
    tag = '  ← HIGH MATCH' if r['score'] >= config.HIGH_MATCH_THRESHOLD else ''
    print(f"  {r['score']:>8.4f}  {r['authorised_name']}  (UID: {r['uid']}){tag}")

## Step 2 — build_prompts.py

`build_prompts.py` reads `cna_articles.csv`, filters relevant rows, and builds **one self-contained prompt per article × event type**.

Key design: **no entity name is fed in**. The LLM finds the primary entity itself from the article text, guided only by the event type and expected entity type. Fields to extract come from `event_type_to_field_mappings.json` — edit that file to change extraction behaviour without touching code.

Output: `outputs/prompts_for_batch.csv`

In [ ]:
result = subprocess.run(
    ['python', 'build_prompts.py'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr[:600])

## Step 3 — LLM responses (pre-done)

Each prompt is sent to an LLM externally. The LLM returns:
- `entity_confirmed` — is the article actually about a qualifying Singapore entity?
- `entity_name` — most specific official name found in the article
- `entity_type` — PERSON / ORGANISATION / FACILITY / etc.
- `fields` — extracted values with exact evidence sentences
- `confidence` — low / medium / high

Responses are in `data/fact_extraction.csv` as column `extracted_entities`. **This step is already done.**

## Step 4 — join_responses.py

For each row: parse LLM JSON → check `entity_confirmed` → BM25 link entity to TTE → look up current TTE field value → determine merge state → write to review queue.

Merge states: `ADD` (field empty in TTE), `APPEND` (pipe-delimited list, append new value), `REPLACE` (single value, overwrite), `DUPLICATE` (value already in TTE — skip).

Note: `join_responses.py` reads `outputs/prompts_for_batch.csv` with column `llm_response`. Our responses are in `data/fact_extraction.csv` as `extracted_entities` — the cell below handles the rename.

In [ ]:
# Bridge: copy fact_extraction.csv → prompts_for_batch.csv with column renamed
fe = pd.read_csv('../data/fact_extraction.csv', dtype=str).fillna('')
fe.rename(columns={'extracted_entities': 'llm_response'}).to_csv(
    '../outputs/prompts_for_batch.csv', index=False
)
print(f'Wrote {len(fe):,} rows to outputs/prompts_for_batch.csv')
print()

result = subprocess.run(
    ['python', 'join_responses.py'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr[:600])

---

## Walkthrough: one article, all the way through

Now that all four CSVs exist, let's follow a single article through every step — no API calls, just reading what was already computed.

**Article:** *Urban Redevelopment Authority to get new CEO; next chief planner appointed*  
**Event type:** person appointed or elected

In [ ]:
# The row_id that ties this article across all CSVs
TRACE_ID = '14_person_appointed_or_elect'

fe_all  = pd.read_csv('../data/fact_extraction.csv', dtype=str).fillna('')
fe_row  = fe_all[fe_all['row_id'] == TRACE_ID].iloc[0]
art_row = articles[articles['URL'] == fe_row['article_url']].iloc[0]

print('STEP 0 — Article')
print('─' * 68)
print(f"Title      : {art_row['Title']}")
print(f"URL        : {art_row['URL']}")
print()
print(f"Text (first 500 chars):")
print(art_row['text'][:500], '...')

In [ ]:
print('STEP 0 — Classifier decision (Prompt 1, already done)')
print('─' * 68)
print(f"Event type : {art_row['Event_Type']}")
print(f"Reasoning  : {art_row['Reasoning']}")
print()
print('The classifier only saw the title. Cheap gate — no full text needed.')

In [ ]:
print('STEP 2 — Fact extraction prompt (Prompt 2)')
print('─' * 68)
print(fe_row['prompt'])
print()
print('Note: no entity name in the prompt. The LLM finds it from the article text.')

In [ ]:
print('STEP 3 — LLM response')
print('─' * 68)
parsed = json.loads(fe_row['extracted_entities'])
print(json.dumps(parsed, indent=2))

In [ ]:
print('STEP 4 — join_responses.py output')
print('─' * 68)

queue = pd.read_csv('../outputs/review_queue.csv', dtype=str).fillna('')
trace_rows = queue[queue['row_id'] == TRACE_ID]

print(f"Entity matched : {trace_rows.iloc[0]['tte_authorised_name']}")
print(f"TTE UID        : {trace_rows.iloc[0]['tte_uid']}")
print(f"Match score    : {trace_rows.iloc[0]['match_score']}")
print(f"Match type     : {trace_rows.iloc[0]['match_type']}")
print()
print(f"{'Field':<28}  {'Merge':<9}  {'Current value':<30}  New value")
print('─' * 110)
for _, r in trace_rows.iterrows():
    current = (r['current_value'][:27] + '...') if len(r['current_value']) > 30 else r['current_value']
    print(f"  {r['field']:<26}  {r['merge_state']:<9}  {current or '(empty)':<30}  {r['new_value'][:60]}")

In [ ]:
print('STEP 5 — What the reviewer sees (one card in review_app.py)')
print('=' * 68)

r0 = trace_rows.iloc[0]
name = r0['tte_authorised_name']
print(f"  {name}  [{r0['tte_vocabulary']}]")
print(f"  UID: {r0['tte_uid']}  |  Score: {r0['match_score']}  |  {r0['record_richness']} TTE fields")
print('=' * 68)
print(f"  Event type : {r0['event_type']}")
print(f"  Article    : {r0['article_title'][:65]}")
if r0['entity_name'] != r0['tte_authorised_name']:
    print(f"  ⚠  Extracted as '{r0['entity_name']}' → matched to '{r0['tte_authorised_name']}'")
print()

MERGE_ACTION = {'ADD': 'Add new field', 'APPEND': 'Append to existing', 'REPLACE': 'Replace existing'}
for _, r in trace_rows.iterrows():
    action = MERGE_ACTION.get(r['merge_state'], r['merge_state'])
    print(f"  TTE Field  : {r['field']}")
    print(f"  Current    : {r['current_value'] or '(empty)'}")
    print(f"  New value  : {r['new_value']}   [action: {action}]")
    if r['merge_state'] == 'APPEND':
        print(f"  After      : {r['final_value']}")
    print(f"  Evidence   : \"{r['evidence'][:120]}\"")
    print()

print('  [✅ Approve All]  [❌ Reject]  [⏭ No KB Update]  [🚩 Flag]')
print('─' * 68)

---

## Review queue — full dataset

In [ ]:
print(f'review_queue.csv — {len(queue):,} rows\n')

print('By event type (top 10):')
print(queue['event_type'].value_counts().head(10).to_string())
print()

print('By merge state:')
print(queue['merge_state'].value_counts().to_string())
print()

print('By match type:')
print(queue['match_type'].value_counts().to_string())

In [ ]:
# New entity candidates — entities the LLM found that aren't in TTE yet
cands = pd.read_csv('../outputs/new_entity_candidates.csv', dtype=str).fillna('')

print(f'new_entity_candidates.csv — {len(cands):,} rows\n')
print('By entity type:')
print(cands['entity_type'].value_counts().to_string())
print()

print('Sample candidates:')
cols = ['entity_name', 'entity_type', 'event_type', 'article_title']
pd.set_option('display.max_colwidth', 50)
print(cands[cols].head(8).to_string(index=False))

## Step 5 — Human review

Run from the project root:

```bash
streamlit run review_app.py
```

Four pages:

| Page | What you do |
|------|-------------|
| **Dashboard** | Pending counts, event type breakdown, recent activity |
| **Event Updates** | Review proposed field changes — edit values, choose action (Append / Replace / Add), approve / reject / flag |
| **New Entities** | Review no-match entities — one-click OneSearch verification, flag for research or dismiss |
| **History & Backup** | Download `approved_changes.csv` in TTE import format |

Each decision is saved to `outputs/decisions.json` with reviewer name, timestamp, and any edited values. Decisions can be undone.

**Export format** (History & Backup → download button):

```
Key UID | Key Descriptor | Key Vocabulary | Relationship Type | Related Descriptor | Evidence | Article URL | Reviewer | Timestamp
```

One row per approved field change. If the reviewer edited the extracted value, the edited version appears in Related Descriptor.

---

## Summary

```
2,970 CNA articles scraped
    ↓  Prompt 1: LLM classifier on title only  [done]
~420 relevant articles (14%)
    ↓  python build_prompts.py
~730 prompts  (one per article × event type)
    ↓  Prompt 2: LLM fact extraction  [done — data/fact_extraction.csv]
    ↓  python join_responses.py
~870 proposed field changes  →  outputs/review_queue.csv
~130 new entity candidates   →  outputs/new_entity_candidates.csv
    ↓  streamlit run review_app.py
Reviewer approves / rejects / flags each card
    ↓  Export
approved_changes.csv  →  manual TTE import
decisions.json        →  audit trail + future training data
```

**Design principles:**
- LLMs used for data generation only — not for final decisions
- Every proposed change is traceable to an exact sentence in the source article
- Precision over recall — false positives waste librarian time
- Human review is not a fallback — it is the point
- No TTE writes without human approval
- The decision log becomes labelled training data for future model fine-tuning

**The files that matter:**
```
data/cna_articles.csv                  ← scraped + classified articles
data/fact_extraction.csv               ← LLM responses (already done)
event_type_to_field_mappings.json      ← edit to change fields extracted
event_type_to_entity_mappings.json     ← edit to change entity type mapping
01_build_index.py                      ← run once per TTE data update
build_prompts.py                       ← Step 2
join_responses.py                      ← Step 4
review_app.py                          ← Steps 5–8
```